### Save MRMS data for different variables and date ranges

### Begin home location to MRMS matching

#### Load home location file

### Load MRMS variable files

### Match home locations to nearest MRMS data

In [1]:
from plotly.subplots import make_subplots
import plotly
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
plotly.offline.init_notebook_mode(connected=True)


In [78]:
with open("../config/zips_counties.yaml", 'r') as stream:
    out = yaml.load(stream, Loader=yaml.FullLoader)
    zips = [str(z) for z in out["zips_LA"] + out["zips_SB"] + out["zips_VE"]]

In [80]:
mrms_fn = "/glade/campaign/cisl/aiml/cat_flood/mrms_data/2024/veraset_statistics_mrms_20250721.parquet"
mrms_df = pd.read_parquet(mrms_fn)
mrms_df = mrms_df[mrms_df.zipcode.isin(zips)]
print(mrms_df.shape)

(5024, 13)


In [81]:
data_dir = "/glade/campaign/cisl/aiml/cat_flood/analysis_ready_data_20250720/"
fns = [f"veraset_2024-03-{date:02}_PST_chunked.parquet" for date in range(29,32)]
fns += [f"veraset_2024-04-{date:02}_PST_chunked.parquet" for date in range(1,2)]
metrics_df = pd.concat(
    pd.read_parquet(data_dir+pf)
    for pf in fns)
metrics_df = metrics_df[metrics_df.zipcode.isin(zips)]
print(metrics_df.shape)

(16989, 16)


In [82]:
matching_uids = np.intersect1d(metrics_df.uid, mrms_df.index)
matching_uids.shape

(4483,)

In [9]:
def get_precip_category(uid, mrms_df, variable):
    high_uids = mrms_df[mrms_df[variable] == 'high'].index
    if uid in high_uids:
        return 'high'
    else:
        return 'low'

In [106]:
metrics_df.groupby(mrms_metric_cat).size()

precip_sum_category
low    16989
dtype: int64

In [107]:
# 90-10 percentile thresholds (while tracking n users)
# people with no precip vs people who experienced more than 0mm precip and work way up to 50-50
# pare down to the three counties of interest
# add split between three days in AR period and three days in non-AR period with same days of week
# max stream flow instead of precip data

mrms_metrics = ["precip_max", "precip_sum"]
mrms_labels = ["Max Precipitation (MRMS)", "Cumulative Precipitation (MRMS)"]
mobility_metrics = ['number_of_visits', 'distance_straight_line',
                    'radius_of_gyration', 'num_stops', 'max_distance_from_home']
mobility_labels = ['Number of Visits ', 'Cumulative Distance',
                   'Radius of Gyration', 'Number of Stops', 'Maxmimum Distance from Home']
percentiles = np.arange(0.1, 1, 0.1)
percentile_keys = [f"{int(p*100)}%" for p in percentiles]

for perc in percentile_keys:
    for i, mrms_metric in enumerate(mrms_metrics[1:2]):
        metric_describe = mrms_df[mrms_metric].describe(percentiles=percentiles)
        mrms_metric_cat = f"{mrms_metric}_category"
        mrms_df[mrms_metric_cat] = pd.cut(mrms_df[mrms_metric], bins=[0, metric_describe[perc], float('Inf')], labels=['low', 'high'])
        metrics_df[mrms_metric_cat] = metrics_df.uid.apply(get_precip_category, mrms_df=mrms_df, variable=mrms_metric_cat)
        for j, mobility_metric in enumerate(mobility_metrics):
            fig = make_subplots(rows=2, cols=2,
                                specs=[[{"colspan": 2}, None],
                                       [{}, {}]],
                subplot_titles=(f"<span style='font-weight:bold;'>{mrms_labels[i]}</span>", "", ""),
                shared_yaxes=True,
                horizontal_spacing=0.01,
                vertical_spacing=0.1)
            
            fig.add_trace(
                go.Box(x=mrms_df[mrms_df[mrms_metric_cat] == "high"][mrms_metric_cat],
                       y=mrms_df[mrms_df[mrms_metric_cat] == "high"][mrms_metric],
                       marker_color='#FF4136', showlegend=False),
                row=1, col=1
            )
            fig.add_trace(
                go.Box(x=mrms_df[mrms_df[mrms_metric_cat] == "low"][mrms_metric_cat],
                       y=mrms_df[mrms_df[mrms_metric_cat] == "low"][mrms_metric],
                       marker_color='#3D9970', showlegend=False),
                row=1, col=1
            )
            fig.add_trace(
                go.Box(x=metrics_df[metrics_df[mrms_metric_cat] == "high"]["date"],
                       y=metrics_df[metrics_df[mrms_metric_cat] == "high"][mobility_metric],
                       marker_color='#FF4136', showlegend=False),
                row=2, col=1
            )
            fig.add_trace(
                go.Box(x=metrics_df[metrics_df[mrms_metric_cat] == "low"]["date"],
                       y=metrics_df[metrics_df[mrms_metric_cat] == "low"][mobility_metric],
                       marker_color='#3D9970', showlegend=False),
                row=2, col=2
            )
    
            fig.add_annotation(text=f"<b>{mobility_labels[j]}<b>",
                               xref="paper", yref="paper",
                               x=0.5, y=-0.13, showarrow=False
                               )
            fig.update_annotations(font_size=18)
        
            fig.update_layout(font_size=16,
                              autosize=False,
                              width=800,
                              height=600,
                              margin=dict(l=20,r=20,b=70,t=40),
                              )
            fig.write_html(f"html/mrms/plotly_{mrms_metric}_{mobility_metric}_{perc}.html")
            fig.show()